# Prueba Técnica Digotec — Análisis y Segmentación de Cartera Bancaria

**Autor:** Daniel Coello  
**Fecha:** Septiembre 2026  
**Dataset:** `Documentacion/Digotec_Prueba_Analitica_Automatizacion_Dataset.tsv`

---

## Objetivo

Transformar un dataset bancario crudo en una base confiable y analítica que permita:
1. Entender la cartera de clientes (productos, saldos, consumos)
2. Identificar segmentos de comportamiento ('Lovers')
3. Detectar insights accionables para el negocio

## Flujo del Notebook
```
TSV crudo → Carga → Exploración → Limpieza → Vista por cliente → Lovers → Insights → Exportar
```

---
## 0. Configuración e Importaciones

Importamos `pandas` para manipular datos y `warnings` para suprimir mensajes irrelevantes.  
`pathlib.Path` nos permite construir rutas de archivos de forma compatible con cualquier sistema operativo.

In [1]:
import pandas as pd
import warnings
from pathlib import Path
from datetime import datetime, date

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Rutas base del proyecto
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'Documentacion'
OUTPUT_DIR = BASE_DIR / 'data'
OUTPUT_DIR.mkdir(exist_ok=True)

print('✅ Librerías cargadas correctamente')
print(f'📁 Output en: {OUTPUT_DIR.resolve()}')

✅ Librerías cargadas correctamente
📁 Output en: C:\Users\Daniel Coello\Documents\GitHub\DIGOTEC\data


---
## 1. Carga del Dataset

El archivo viene en formato **TSV** (Tab-Separated Values): igual que un CSV pero separado por tabulaciones en vez de comas.  
Usamos `encoding='utf-8-sig'` para manejar correctamente caracteres especiales (tildes, ñ) que son comunes en datos bancarios latinoamericanos.  
El parámetro `errors='replace'` evita que el script explote si hay un caracter no reconocido.

In [2]:
raw_df = pd.read_csv(
    DATA_DIR / 'Digotec_Prueba_Analitica_Automatizacion_Dataset.tsv',
    sep='\t',
    encoding='utf-8-sig',
    encoding_errors='replace'
)

print(f'📊 Filas cargadas:    {len(raw_df):,}')
print(f'📋 Columnas:          {raw_df.shape[1]}')
print(f'👤 Clientes únicos:   {raw_df["cliente_id"].nunique():,}')
print()
raw_df.head(5)

📊 Filas cargadas:    22,455
📋 Columnas:          15
👤 Clientes únicos:   2,200



,cliente_id,nombre_cliente,segmento_cliente,ciudad,edad,ingreso_estimado,producto,saldo_producto,cupo_credito,fecha_vencimiento,fecha_movimiento,categoria_consumo,monto_consumo,canal,estado_producto
0,C000728,Cliente 0728,Mass,Loja,50,1019.90,Cuenta de Ahorros,1498.82,NaN,NaN,2026-01-15,NaN,0.00,Branch,Activo
1,C001264,Cliente 1264,Mass,Guayaquil,57,471.10,Crédito Vehicular,20725.80,NaN,2029-11-14,2026-03-20,NaN,0.00,App,Activo
2,C001515,Cliente 1515,Joven,Manta,45,639.34,Tarjeta de Crédito,689.27,1200.00,2029-03-16,2025-11-06,Food,16.63,Branch,Activo
3,C002171,Cliente 2171,Mass,Guayaquil,23,1159.30,Cuenta de Ahorros,5001.88,NaN,NaN,2025-06-09,NaN,0.00,App,Activo
4,C001848,Cliente 1848,Premium,Guayaquil,46,3037.26,Tarjeta de Crédito,1955.00,8200.00,2028-12-28,2025-03-04,Food,51.91,App,Activo


---
## 2. Exploración Inicial — Conocer los datos antes de tocarlos

**Regla de oro del análisis de datos:** Nunca limpies algo que no entiendes primero.  
Aquí revisamos:
- Tipos de datos que infirió pandas
- Cuántos valores nulos hay por columna
- Los valores únicos de columnas categóricas (donde suelen estar los problemas)

In [3]:
print('=== TIPOS DE DATOS ===')
print(raw_df.dtypes)
print()
print('=== VALORES NULOS / VACÍOS POR COLUMNA ===')
nulls = raw_df.isnull().sum().to_frame('nulos') 
nulls['vacios_str'] = (raw_df == '').sum()
nulls['total_problemas'] = nulls['nulos'] + nulls['vacios_str']
nulls['% del total'] = (nulls['total_problemas'] / len(raw_df) * 100).round(1)
print(nulls[nulls['total_problemas'] > 0])

=== TIPOS DE DATOS ===
cliente_id               str
nombre_cliente           str
segmento_cliente         str
ciudad                   str
edad                   int64
ingreso_estimado     float64
producto                 str
saldo_producto       float64
cupo_credito         float64
fecha_vencimiento        str
fecha_movimiento         str
categoria_consumo        str
monto_consumo        float64
canal                    str
estado_producto          str
dtype: object

=== VALORES NULOS / VACÍOS POR COLUMNA ===
                   nulos  vacios_str  total_problemas  % del total
ciudad               178           0              178         0.80
saldo_producto       136           0              136         0.60
cupo_credito        7513           0             7513        33.50
fecha_vencimiento   3464           0             3464        15.40
categoria_consumo   7513           0             7513        33.50


In [4]:
# Revisamos los valores únicos de las columnas categóricas que sabemos que tienen problemas
print('=== segmento_cliente (valores únicos) ===')
print(sorted(raw_df['segmento_cliente'].dropna().unique()))

print('\n=== categoria_consumo (valores únicos) ===')
print(sorted(raw_df['categoria_consumo'].dropna().unique()))

print('\n=== producto (valores únicos) ===')
print(sorted(raw_df['producto'].dropna().unique()))

print('\n=== canal (valores únicos) ===')
print(sorted(raw_df['canal'].dropna().unique()))

print('\n=== ciudad (valores únicos) ===')
print(sorted(raw_df['ciudad'].fillna('(vacío)').unique()))

=== segmento_cliente (valores únicos) ===
['Affluent', 'Affluent ', 'JOVEN', 'Joven', 'Mass', 'PREMIUM ', 'Premium', 'PyME', 'mass', 'pyme']

=== categoria_consumo (valores únicos) ===
['Education', 'Entertainment', 'FOOD ', 'Food', 'Fuel', 'Health', 'Others', 'Streaming', 'Super Market', 'Supermarket', 'Technology', 'Travel', 'Travels', 'streamng', 'tech']

=== producto (valores únicos) ===
['Crédito Hipotecario', 'Crédito Vehicular', 'Crédito de Consumo', 'Cuenta Corriente', 'Cuenta de Ahorros', 'Tarjeta de Crédito']

=== canal (valores únicos) ===
['ATM', 'App', 'Branch', 'POS', 'Web']

=== ciudad (valores únicos) ===
['(vacío)', 'Ambato', 'Cuenca', 'Guayaquil', 'Loja', 'Manta', 'Quito']


### Hallazgos de la exploración

| Problema | Campo | Ejemplo | Decisión |
|---|---|---|---|
| Inconsistencia de mayúsculas | `segmento_cliente` | `"JOVEN"`, `"Joven"`, `"joven"` | Normalizar a Title Case con mapeo explícito |
| Espacios extra | `segmento_cliente` | `"Affluent "` (espacio al final) | `.strip()` |
| Typos en categorías | `categoria_consumo` | `"streamng"`, `"tech"`, `"Travels"` | Mapeo de corrección |
| Ciudades vacías | `ciudad` | `""` (178 registros) | Imputar como `"Desconocida"` |
| Nulos esperados | `fecha_vencimiento` | Solo Tarjetas y Créditos tienen vencimiento | Mantener nulo, es correcto por negocio |
| Nulos esperados | `cupo_credito` | Solo Tarjetas de Crédito tienen cupo | Mantener nulo, es correcto por negocio |
| Filas de tipo N/A | `categoria_consumo` | Productos sin tarjeta no tienen categoría de consumo | Mantener N/A, es correcto |

---
## 3. Limpieza de Datos

Trabajamos sobre una **copia** del DataFrame original (`df`).  
Esto es buena práctica: si nos equivocamos, siempre podemos volver a `raw_df` sin volver a leer el archivo.

In [5]:
df = raw_df.copy()

# ─────────────────────────────────────────────────────────────
# 3.1 NORMALIZAR segmento_cliente
# Estrategia: strip() elimina espacios, luego mapeamos cada variante
# al valor canónico del negocio.
# ─────────────────────────────────────────────────────────────
SEGMENTO_MAP = {
    'mass': 'Mass',
    'joven': 'Joven',
    'premium': 'Premium',
    'affluent': 'Affluent',
    'pyme': 'PyME',
}

df['segmento_cliente'] = (
    df['segmento_cliente']
    .str.strip()                    # quita espacios al inicio y final
    .str.lower()                    # todo a minúsculas para poder mapear
    .map(SEGMENTO_MAP)              # reemplaza según el diccionario
    .fillna('Desconocido')          # si algo no matchea, queda 'Desconocido'
)

print('Segmentos después de limpiar:', sorted(df['segmento_cliente'].unique()))

Segmentos después de limpiar: ['Affluent', 'Joven', 'Mass', 'Premium', 'PyME']


In [6]:
# ─────────────────────────────────────────────────────────────
# 3.2 NORMALIZAR categoria_consumo
# Mapeamos variantes sucias a su forma canónica.
# Los 'N/A' son válidos — corresponden a productos sin tarjeta.
# ─────────────────────────────────────────────────────────────
CATEGORIA_MAP = {
    'food': 'Food',
    'food ': 'Food',
    'supermarket': 'Supermarket',
    'super market': 'Supermarket',
    'technology': 'Technology',
    'tech': 'Technology',
    'travel': 'Travel',
    'travels': 'Travel',
    'streaming': 'Streaming',
    'streamng': 'Streaming',
    'entertainment': 'Entertainment',
    'health': 'Health',
    'education': 'Education',
    'fuel': 'Fuel',
    'others': 'Others',
    'n/a': 'N/A',
}

df['categoria_consumo'] = (
    df['categoria_consumo']
    .str.strip()
    .str.lower()
    .map(CATEGORIA_MAP)
    .fillna('N/A')
)

print('Categorías después de limpiar:', sorted(df['categoria_consumo'].unique()))

Categorías después de limpiar: ['Education', 'Entertainment', 'Food', 'Fuel', 'Health', 'N/A', 'Others', 'Streaming', 'Supermarket', 'Technology', 'Travel']


In [7]:
# ─────────────────────────────────────────────────────────────
# 3.3 CIUDAD — rellenar vacíos
# Decisión de negocio: preferimos 'Desconocida' a eliminar la fila.
# Esos 178 clientes siguen siendo válidos para el análisis de saldos.
# ─────────────────────────────────────────────────────────────
df['ciudad'] = df['ciudad'].str.strip().replace('', 'Desconocida').fillna('Desconocida')

print('Ciudades:', sorted(df['ciudad'].unique()))

Ciudades: ['Ambato', 'Cuenca', 'Desconocida', 'Guayaquil', 'Loja', 'Manta', 'Quito']


In [8]:
# ─────────────────────────────────────────────────────────────
# 3.4 FECHAS — parsear a datetime
# errors='coerce' convierte fechas inválidas o vacías a NaT (Not a Time),
# que es el equivalente de NaN para fechas. Así no se rompe el proceso.
# ─────────────────────────────────────────────────────────────
df['fecha_vencimiento'] = pd.to_datetime(df['fecha_vencimiento'], errors='coerce')
df['fecha_movimiento']  = pd.to_datetime(df['fecha_movimiento'],  errors='coerce')

# Campo calculado: días hasta el vencimiento (desde hoy)
hoy = pd.Timestamp(date.today())
df['dias_para_vencimiento'] = (df['fecha_vencimiento'] - hoy).dt.days

print(f'Productos con fecha de vencimiento: {df["fecha_vencimiento"].notna().sum():,}')
print(f'Productos SIN fecha (cuentas, normal): {df["fecha_vencimiento"].isna().sum():,}')

Productos con fecha de vencimiento: 18,991
Productos SIN fecha (cuentas, normal): 3,464


In [9]:
# ─────────────────────────────────────────────────────────────
# 3.5 NORMALIZAR producto (hay caracteres corruptos por encoding)
# ─────────────────────────────────────────────────────────────
PRODUCTO_MAP = {
    lambda x: 'tarjeta' in str(x).lower(): 'Tarjeta de Crédito',
}

# Limpieza manual de nombres corruptos
df['producto'] = df['producto'].str.strip()
df['producto'] = df['producto'].replace({
    'Cr\x00\x00dito Vehicular': 'Crédito Vehicular',
    'Cr\x00\x00dito Hipotecario': 'Crédito Hipotecario',
    'Cr\x00\x00dito de Consumo': 'Crédito de Consumo',
    'Tarjeta de Cr\x00\x00dito': 'Tarjeta de Crédito',
})
# Normalización robusta: si contiene la palabra clave, normalizamos
df.loc[df['producto'].str.contains('arjeta', na=False), 'producto'] = 'Tarjeta de Crédito'
df.loc[df['producto'].str.contains('Vehicular', na=False), 'producto'] = 'Crédito Vehicular'
df.loc[df['producto'].str.contains('Hipotecario', na=False), 'producto'] = 'Crédito Hipotecario'
df.loc[df['producto'].str.contains('Consumo', na=False), 'producto'] = 'Crédito de Consumo'
df.loc[df['producto'].str.contains('Ahorros', na=False), 'producto'] = 'Cuenta de Ahorros'
df.loc[df['producto'].str.contains('Corriente', na=False), 'producto'] = 'Cuenta Corriente'

print('Productos normalizados:', sorted(df['producto'].unique()))

Productos normalizados: ['Crédito Hipotecario', 'Crédito Vehicular', 'Crédito de Consumo', 'Cuenta Corriente', 'Cuenta de Ahorros', 'Tarjeta de Crédito']


In [10]:
# ─────────────────────────────────────────────────────────────
# 3.6 TIPOS NUMÉRICOS
# pandas pudo haber cargado algunos campos como string.
# Convertimos explícitamente los que usaremos en cálculos.
# ─────────────────────────────────────────────────────────────
df['monto_consumo']  = pd.to_numeric(df['monto_consumo'],  errors='coerce').fillna(0)
df['saldo_producto'] = pd.to_numeric(df['saldo_producto'], errors='coerce').fillna(0)
df['cupo_credito']   = pd.to_numeric(df['cupo_credito'],   errors='coerce')
df['ingreso_estimado'] = pd.to_numeric(df['ingreso_estimado'], errors='coerce')
df['edad']           = pd.to_numeric(df['edad'],           errors='coerce')

print('✅ Tipos numéricos corregidos')

✅ Tipos numéricos corregidos


In [11]:
# ─────────────────────────────────────────────────────────────
# 3.7 VERIFICACIÓN FINAL DE LIMPIEZA
# ─────────────────────────────────────────────────────────────
print('=== RESUMEN POST-LIMPIEZA ===')
print(f'Filas totales:         {len(df):,}')
print(f'Clientes únicos:       {df["cliente_id"].nunique():,}')
print(f'Filas duplicadas:      {df.duplicated().sum():,}')
print()
print('Segmentos:  ', df['segmento_cliente'].value_counts().to_dict())
print('Categorías: ', df['categoria_consumo'].value_counts().to_dict())
print('Productos:  ', df['producto'].value_counts().to_dict())

=== RESUMEN POST-LIMPIEZA ===
Filas totales:         22,455


Clientes únicos:       2,200
Filas duplicadas:      113

Segmentos:   {'Mass': 10018, 'Affluent': 4450, 'Premium': 3104, 'Joven': 2788, 'PyME': 2095}
Categorías:  {'N/A': 7513, 'Supermarket': 2239, 'Food': 2207, 'Travel': 1764, 'Streaming': 1758, 'Technology': 1690, 'Entertainment': 1368, 'Others': 1031, 'Health': 995, 'Education': 947, 'Fuel': 943}
Productos:   {'Tarjeta de Crédito': 14942, 'Cuenta de Ahorros': 1741, 'Cuenta Corriente': 1723, 'Crédito de Consumo': 1439, 'Crédito Hipotecario': 1378, 'Crédito Vehicular': 1232}


---
## 4. Vista Analítica por Cliente

El dataset tiene **1 fila por producto** (un cliente puede aparecer varias veces si tiene varios productos).  
Necesitamos **1 fila por cliente** para poder segmentar, graficar y cargar en el frontend.

Usamos `groupby('cliente_id')` que agrupa todas las filas del mismo cliente y luego calculamos métricas agregadas.

**¿Por qué esta transformación?**  
Porque las preguntas de negocio son a nivel cliente:  
- *¿Cuántos clientes tenemos?* → no filas  
- *¿Cuánto saldo tiene un cliente?* → suma de todos sus productos  
- *¿Qué Lover es?* → basado en su patrón de consumo total

In [12]:
# ─────────────────────────────────────────────────────────────
# 4.1 CONSUMO POR CLIENTE Y CATEGORÍA (solo Tarjetas de Crédito)
# Esto nos dirá: ¿en qué categoría gasta MÁS cada cliente?
# ─────────────────────────────────────────────────────────────
tc_df = df[df['producto'] == 'Tarjeta de Crédito'].copy()
tc_df = tc_df[tc_df['categoria_consumo'] != 'N/A']

# Suma de monto_consumo por cliente y categoría
consumo_por_cat = (
    tc_df.groupby(['cliente_id', 'categoria_consumo'])['monto_consumo']
    .sum()
    .reset_index()
)

# Para cada cliente, encontramos la categoría con MAYOR consumo
cat_dominante = (
    consumo_por_cat
    .sort_values('monto_consumo', ascending=False)
    .groupby('cliente_id')
    .first()
    .reset_index()
    .rename(columns={'categoria_consumo': 'categoria_dominante', 'monto_consumo': 'consumo_dominante'})
)

print(f'Clientes con Tarjeta de Crédito y consumo registrado: {len(cat_dominante):,}')
cat_dominante.head()

Clientes con Tarjeta de Crédito y consumo registrado: 1,655


,cliente_id,categoria_dominante,consumo_dominante
0,C000001,Technology,1231.50
1,C000002,Fuel,74.31
2,C000003,Food,456.69
3,C000006,Technology,73.84
4,C000007,Travel,521.41


In [13]:
# ─────────────────────────────────────────────────────────────
# 4.2 ASIGNAR LOVER TYPE
# Basado en la categoría dominante de consumo.
# Criterio de negocio: agrupamos categorías similares para crear
# segmentos más amplios y accionables comercialmente.
# ─────────────────────────────────────────────────────────────
LOVER_MAP = {
    'Food':          'Food & Supermarket Lover',
    'Supermarket':   'Food & Supermarket Lover',
    'Technology':    'Tech Lover',
    'Travel':        'Travel Lover',
    'Streaming':     'Entertainment & Streaming Lover',
    'Entertainment': 'Entertainment & Streaming Lover',
    'Health':        'Health & Wellness Lover',
    'Education':     'Education Lover',
    'Fuel':          'Lifestyle Lover',
    'Others':        'Lifestyle Lover',
}

cat_dominante['lover_type'] = cat_dominante['categoria_dominante'].map(LOVER_MAP)

print('Distribución de Lovers:')
print(cat_dominante['lover_type'].value_counts())

Distribución de Lovers:
lover_type
Food & Supermarket Lover           529
Entertainment & Streaming Lover    326
Travel Lover                       229
Tech Lover                         205
Lifestyle Lover                    188
Health & Wellness Lover             89
Education Lover                     89
Name: count, dtype: int64


In [14]:
# ─────────────────────────────────────────────────────────────
# 4.3 AGREGAR MÉTRICAS BASE POR CLIENTE
# ─────────────────────────────────────────────────────────────
clients_base = df.groupby('cliente_id').agg(
    nombre_cliente     = ('nombre_cliente',      'first'),
    segmento_cliente   = ('segmento_cliente',    'first'),
    ciudad             = ('ciudad',              'first'),
    edad               = ('edad',                'first'),
    ingreso_estimado   = ('ingreso_estimado',    'first'),
    total_productos    = ('producto',            'count'),
    lista_productos    = ('producto',            lambda x: ', '.join(sorted(x.unique()))),
    saldo_total        = ('saldo_producto',      'sum'),
    consumo_total_tc   = ('monto_consumo',       'sum'),
    cupo_total_tc      = ('cupo_credito',        'sum'),
    dias_venc_minimo   = ('dias_para_vencimiento', 'min'),
    canales_usados     = ('canal',               lambda x: ', '.join(sorted(x.unique()))),
).reset_index()

# Flag: cliente multiproducto (2 o más productos distintos)
clientes_productos_distintos = df.groupby('cliente_id')['producto'].nunique().reset_index()
clientes_productos_distintos.columns = ['cliente_id', 'n_productos_distintos']
clients_base = clients_base.merge(clientes_productos_distintos, on='cliente_id')
clients_base['es_multiproducto'] = clients_base['n_productos_distintos'] >= 2

# Porcentaje de utilización de Tarjeta de Crédito
# Decisión: si no tiene cupo registrado (no tiene TC), queda NaN
clients_base['utilizacion_tc_pct'] = (
    (clients_base['consumo_total_tc'] / clients_base['cupo_total_tc'] * 100)
    .round(1)
)

print(f'Vista cliente generada: {len(clients_base):,} filas')
clients_base.head(3)

Vista cliente generada: 2,200 filas


,cliente_id,nombre_cliente,segmento_cliente,ciudad,edad,ingreso_estimado,total_productos,lista_productos,saldo_total,consumo_total_tc,cupo_total_tc,dias_venc_minimo,canales_usados,n_productos_distintos,es_multiproducto,utilizacion_tc_pct
0,C000001,Cliente 0001,Affluent,Guayaquil,37,5885.36,11,"Crédito de Consumo, Tarjeta de Crédito",39838.74,3384.12,62000.00,405.00,"App, Branch, POS, Web",2,True,5.50
1,C000002,Cliente 0002,PyME,Loja,55,651.61,8,"Crédito Hipotecario, Tarjeta de Crédito",221386.40,211.33,4000.00,1017.00,"App, Branch, POS, Web",2,True,5.30
2,C000003,Cliente 0003,Affluent,Manta,31,2471.02,7,Tarjeta de Crédito,14949.83,1040.90,46200.00,636.00,"App, Branch, POS, Web",1,False,2.30


In [15]:
# ─────────────────────────────────────────────────────────────
# 4.4 UNIR LOVERS CON LA VISTA DE CLIENTES
# ─────────────────────────────────────────────────────────────
clients_df = clients_base.merge(cat_dominante[['cliente_id', 'lover_type', 'categoria_dominante']], 
                                 on='cliente_id', how='left')

# Clientes sin Tarjeta de Crédito no tienen Lover type
clients_df['lover_type'] = clients_df['lover_type'].fillna('Sin Perfil de Consumo')
clients_df['categoria_dominante'] = clients_df['categoria_dominante'].fillna('N/A')

print('=== DISTRIBUCIÓN FINAL DE LOVERS ===')
print(clients_df['lover_type'].value_counts())
print(f'\nTotal clientes: {len(clients_df):,}')

=== DISTRIBUCIÓN FINAL DE LOVERS ===
lover_type
Sin Perfil de Consumo              545
Food & Supermarket Lover           529
Entertainment & Streaming Lover    326
Travel Lover                       229
Tech Lover                         205
Lifestyle Lover                    188
Health & Wellness Lover             89
Education Lover                     89
Name: count, dtype: int64

Total clientes: 2,200


---
## 5. Los 3 Insights Accionables

**¿Qué es un insight accionable?**  
No es solo un número. Es una observación que lleva a una **decisión de negocio concreta**.  
Formato: *"X% de los clientes [condición] → esto implica [acción comercial]"*

In [16]:
# ─────────────────────────────────────────────────────────────
# INSIGHT 1: Clientes con tarjeta venciendo en <90 días
# Oportunidad: renovación proactiva antes de que el cliente note el vencimiento
# ─────────────────────────────────────────────────────────────
alertas_df = df[
    (df['producto'] == 'Tarjeta de Crédito') & 
    (df['dias_para_vencimiento'] <= 90) & 
    (df['dias_para_vencimiento'] >= 0)
].copy()

n_alertas = alertas_df['cliente_id'].nunique()
print(f'💡 INSIGHT 1 — Oportunidad de Renovación Proactiva')
print(f'   {n_alertas} clientes tienen tarjetas venciendo en los próximos 90 días.')
print(f'   Acción: campaña de renovación anticipada antes de que el cliente pierda acceso al crédito.')
print()

💡 INSIGHT 1 — Oportunidad de Renovación Proactiva
   107 clientes tienen tarjetas venciendo en los próximos 90 días.
   Acción: campaña de renovación anticipada antes de que el cliente pierda acceso al crédito.



In [17]:
# ─────────────────────────────────────────────────────────────
# INSIGHT 2: Clientes multiproducto vs saldo promedio
# Oportunidad: cross-selling tiene impacto medible en saldo
# ─────────────────────────────────────────────────────────────
saldo_multi  = clients_df[clients_df['es_multiproducto'] == True]['saldo_total'].mean()
saldo_simple = clients_df[clients_df['es_multiproducto'] == False]['saldo_total'].mean()
n_multi = clients_df['es_multiproducto'].sum()
pct_multi = n_multi / len(clients_df) * 100

print(f'💡 INSIGHT 2 — Valor del Cliente Multiproducto')
print(f'   Solo el {pct_multi:.0f}% de los clientes tiene 2+ productos ({n_multi} clientes).')
print(f'   Su saldo promedio (${saldo_multi:,.0f}) es {saldo_multi/saldo_simple:.1f}x mayor que el de un solo producto (${saldo_simple:,.0f}).')
print(f'   Acción: priorizar programas de vinculación de segundo producto en clientes Mass con solo Cuenta de Ahorros.')
print()

💡 INSIGHT 2 — Valor del Cliente Multiproducto
   Solo el 81% de los clientes tiene 2+ productos (1787 clientes).
   Su saldo promedio ($88,657) es 3.2x mayor que el de un solo producto ($27,704).
   Acción: priorizar programas de vinculación de segundo producto en clientes Mass con solo Cuenta de Ahorros.



In [18]:
# ─────────────────────────────────────────────────────────────
# INSIGHT 3: Segmento Joven — bajo ingreso, alta digitalización
# Oportunidad: retención temprana con productos digitales
# ─────────────────────────────────────────────────────────────
joven_df = clients_df[clients_df['segmento_cliente'] == 'Joven']
n_joven = len(joven_df)

# Canales digitales del segmento Joven en el dataset completo
joven_raw = df[df['segmento_cliente'] == 'Joven']
canales_joven = joven_raw['canal'].value_counts(normalize=True) * 100
pct_digital = canales_joven.get('App', 0) + canales_joven.get('Web', 0)

print(f'💡 INSIGHT 3 — Segmento Joven: Retención Temprana Digital')
print(f'   {n_joven} clientes del segmento Joven.')
print(f'   El {pct_digital:.0f}% de sus transacciones ocurren en canales digitales (App + Web).')
print(f'   Su ingreso promedio es ${joven_df["ingreso_estimado"].mean():,.0f}, el más bajo entre segmentos.')
print(f'   Acción: ofrecer Tarjeta de Crédito con cupo inicial bajo y beneficios en Streaming/Tecnología,')
print(f'   aprovechando su preferencia digital para reducir costo de adquisición.')

💡 INSIGHT 3 — Segmento Joven: Retención Temprana Digital
   272 clientes del segmento Joven.
   El 43% de sus transacciones ocurren en canales digitales (App + Web).
   Su ingreso promedio es $718, el más bajo entre segmentos.
   Acción: ofrecer Tarjeta de Crédito con cupo inicial bajo y beneficios en Streaming/Tecnología,
   aprovechando su preferencia digital para reducir costo de adquisición.


---
## 6. Exportación de Resultados

Generamos dos archivos:
1. **`dataset_clean.csv`** — el dataset completo limpio (para Power BI)
2. **`clients_summary.csv`** — 1 fila por cliente (para React frontend y estadísticas rápidas)
3. **`clients_summary.json`** — mismo contenido en JSON (para el frontend React)

In [19]:
# Dataset limpio completo → Power BI
df.to_csv(OUTPUT_DIR / 'dataset_clean.csv', index=False, encoding='utf-8-sig')
print(f'✅ dataset_clean.csv exportado ({len(df):,} filas)')

# Vista por cliente → React + Power BI
clients_df.to_csv(OUTPUT_DIR / 'clients_summary.csv', index=False, encoding='utf-8-sig')
print(f'✅ clients_summary.csv exportado ({len(clients_df):,} filas = 1 por cliente)')

# JSON para React (orient='records' = lista de objetos, compatible con JavaScript)
clients_df.to_json(OUTPUT_DIR / 'clients_summary.json', 
                   orient='records', 
                   force_ascii=False,
                   indent=2,
                   date_format='iso')
print(f'✅ clients_summary.json exportado')

# También exportamos las alertas de vencimiento por separado
alertas_export = alertas_df[['cliente_id', 'nombre_cliente', 'segmento_cliente', 'ciudad', 
                              'producto', 'fecha_vencimiento', 'dias_para_vencimiento', 
                              'cupo_credito', 'saldo_producto']].sort_values('dias_para_vencimiento')
alertas_export.to_json(OUTPUT_DIR / 'alertas_vencimiento.json',
                       orient='records',
                       force_ascii=False,
                       indent=2,
                       date_format='iso')
print(f'✅ alertas_vencimiento.json exportado ({len(alertas_export):,} registros)')

✅ dataset_clean.csv exportado (22,455 filas)
✅ clients_summary.csv exportado (2,200 filas = 1 por cliente)
✅ clients_summary.json exportado
✅ alertas_vencimiento.json exportado (1,004 registros)


In [20]:
# ─────────────────────────────────────────────────────────────
# RESUMEN FINAL — estadísticas para el video y el README
# ─────────────────────────────────────────────────────────────
print('=' * 55)
print('RESUMEN EJECUTIVO — CARTERA DIGOTEC')
print('=' * 55)
print(f'Total clientes únicos:        {len(clients_df):,}')
print(f'Saldo total cartera:          ${clients_df["saldo_total"].sum():,.2f}')
print(f'Saldo promedio por cliente:   ${clients_df["saldo_total"].mean():,.2f}')
print(f'Consumo TC total:             ${clients_df["consumo_total_tc"].sum():,.2f}')
print(f'Clientes multiproducto:       {clients_df["es_multiproducto"].sum():,} ({clients_df["es_multiproducto"].mean()*100:.0f}%)')
print(f'Tarjetas vencen <90 días:     {n_alertas:,}')
print()
print('DISTRIBUCIÓN POR SEGMENTO:')
print(clients_df['segmento_cliente'].value_counts().to_string())
print()
print('DISTRIBUCIÓN DE LOVERS:')
print(clients_df['lover_type'].value_counts().to_string())

RESUMEN EJECUTIVO — CARTERA DIGOTEC
Total clientes únicos:        2,200
Saldo total cartera:          $169,871,166.14
Saldo promedio por cliente:   $77,214.17
Consumo TC total:             $1,871,211.49
Clientes multiproducto:       1,787 (81%)
Tarjetas vencen <90 días:     107

DISTRIBUCIÓN POR SEGMENTO:
segmento_cliente
Mass        1023
Affluent     407
Premium      281
Joven        272
PyME         217

DISTRIBUCIÓN DE LOVERS:
lover_type
Sin Perfil de Consumo              545
Food & Supermarket Lover           529
Entertainment & Streaming Lover    326
Travel Lover                       229
Tech Lover                         205
Lifestyle Lover                    188
Health & Wellness Lover             89
Education Lover                     89


---
## 7. Supuestos y Decisiones Documentadas

| Decisión | Criterio |
|---|---|
| Ciudades vacías → `'Desconocida'` | Preferimos mantener los clientes en el análisis. El campo ciudad no es crítico para las métricas de saldo. |
| `fecha_vencimiento` nula = correcto | Las Cuentas de Ahorros y Corrientes no tienen fecha de vencimiento por naturaleza del producto. |
| `cupo_credito` nulo = correcto | Solo las Tarjetas de Crédito tienen cupo asignado. |
| Lovers basado en categoría dominante | Usamos la categoría de **mayor monto total acumulado**, no la más frecuente, para capturar el mayor valor de gasto. |
| Food + Supermarket = un solo Lover | Comercialmente son el mismo perfil: cliente orientado al consumo cotidiano. |
| Streaming + Entertainment = un solo Lover | Perfil de consumo de ocio digital — mismo tipo de oferta comercial (servicios digitales). |
| Alertas = vencimiento en 0–90 días | Ventana comercial razonable para una campaña de renovación proactiva. |